<!-- REFACTOR-COMMENT: POST_PROCESSING -->
### Refactor Comment (Post-Processing)
- **Purpose**: Compare subsidy allocation levels and subsidy-allocation gaps across scenarios for one run.
- **Primary inputs**: `runs/<run_name>/*/output.csv`.
- **Primary outputs**: metric-summary CSVs, metric-comparison gap plots, and 5-year-average allocation and ad-valorem plots.
- **Refactor role**: Keep the main notebook focused on a 5-year policy view while exposing sensitivity of allocation gaps to the averaging window.


<!-- NOTEBOOK-WORKFLOW -->
## Notebook Workflow
### What This Notebook Does
1. Resolve a policy-assessment run folder and load all scenario `output.csv` tables.
2. Build subsidy summaries for 5, 10, 20, and 50-year average windows.
3. Compare subsidy-allocation gaps across those windows.
4. Generate the remaining allocation and ad-valorem figures from the 5-year average only.

### Inputs
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/*/output.csv`

### Outputs
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_<metric>.csv`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_delta_metric_comparison_<scenario>_vs_<optimal>.png`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/subsidies_delta_pct_metric_comparison_<scenario>_vs_<optimal>.png`
- `project/analysis/post_processing/policy_assessment/runs/<run_name>/img/ad_valorem_ratio_<primary_metric>.csv`

### How To Reuse
- Update `run_name`, optional filters, and the metric configuration in the next cell.


# Analyze subsidy allocation-gap runs


In [ ]:
from pathlib import Path

import pandas as pd
import sys
sys.path.insert(0, "../../../..")

from project.analysis.post_processing.shared.io_runs import load_output_bundle
from project.analysis.post_processing.shared.paths import resolve_run
from project.analysis.post_processing.shared.transforms import (
    keep_scenarios,
    build_indicator_summary,
    parse_subsidy_indicator_index,
    subsidy_summary_by_housing_status,
    subsidy_summary_by_income,
    subsidy_delta_relative_to_reference,
    build_ad_valorem_ratio,
)
from project.analysis.post_processing.shared.plots import (
    plot_indicator_bar_grid,
    plot_subsidies_by_housing_status,
    plot_subsidy_delta_by_housing_status,
    plot_subsidy_delta_pct_by_housing_status,
    plot_ad_valorem_ratio_by_housing_status,
)


In [ ]:
# Run selection
run_name = "subsidies_distortions"
selected_scenarios = None  # Example: ["subsidies_2018", "subsidies_2021", "subsidies_2024"]

# Metric configuration
metric_comparison = {
    "avg_first5": "5y avg",
    "avg_first10": "10y avg",
    "avg_first20": "20y avg",
    "avg_first50": "50y avg",
}
primary_metric = "avg_first5"

indicator_pattern = r"^Subsidies total (.*) \(Million euro\)$"
investment_pattern = r"^Investment total (.*) \(Billion euro\)$"
reference_pattern = None  # Example: r"^Investment total (.*) \(Billion euro\)$"
reference_scale = 1e3

# Scenario display names used in plots and exported tables
scenario_renames = {"Reference": "Package2024"}
scenario_title_labels = {
    "Package2024": "Package 2024",
    "OptimalSubsidies": "Optimal subsidies",
}

# Reference scenario for relative gap plots
optimal_scenario = "OptimalSubsidies"

# Plot controls
same_ylim = True
fontsize = 14

if primary_metric not in metric_comparison:
    raise ValueError(f"primary_metric must be one of {list(metric_comparison)}")


## Total subsidies over time by housing segment

In [ ]:
import re
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

_run_folder = resolve_run("policy_assessment", run_name)
_output_bundle = load_output_bundle(_run_folder)
if scenario_renames:
    _output_bundle = {scenario_renames.get(k, k): v for k, v in _output_bundle.items()}

_segment_pattern = re.compile(r"^Subsidies total (.*) \(Million euro\)$")
_display = lambda name: scenario_title_labels.get(name, name)

# Collect time-series data: segment → scenario → yearly Series
_ts_data = {}  # {scenario: {segment: pd.Series}}
_all_segments = set()

for _scenario, _df in _output_bundle.items():
    _mask = _df.index.to_series().str.match(_segment_pattern.pattern)
    _sub = _df.loc[_mask].copy()
    _sub.index = _sub.index.to_series().str.extract(_segment_pattern.pattern, expand=False)
    # Keep only aggregate rows (exclude income-breakdown rows like "- C1" … "- C5")
    _sub = _sub[~_sub.index.str.contains(r" - C\d$")]
    _year_cols = sorted([c for c in _sub.columns if str(c).isdigit()], key=int)
    _ts = _sub[_year_cols].copy()
    _ts.columns = _ts.columns.astype(int)
    _ts_data[_scenario] = {seg: _ts.loc[seg] for seg in _ts.index}
    _all_segments.update(_ts.index.tolist())

_scenarios = list(_ts_data.keys())

# Layout: Single-family first (row 0), Multi-family second (row 1); 3 columns = [OO, PR, SH]
_status_order = ["Owner-occupied", "Privately rented", "Social-housing"]
_segments_ordered = (
    [f"Single-family - {s}" for s in _status_order if f"Single-family - {s}" in _all_segments]
    + [f"Multi-family - {s}" for s in _status_order if f"Multi-family - {s}" in _all_segments]
)

_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
_scenario_color = {s: _colors[i % len(_colors)] for i, s in enumerate(_scenarios)}
_linestyles = ["-", "--", "-.", ":"]
_scenario_ls = {s: _linestyles[i % len(_linestyles)] for i, s in enumerate(_scenarios)}

ncols = 3
nrows = -(-len(_segments_ordered) // ncols)

# sharey="row" so each housing type (SF / MF) shares its own y-axis scale
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(15, 3.8 * nrows),
    sharey="row",
)
_axes_flat = list(axes.flat) if hasattr(axes, "flat") else [axes]

for ax, seg in zip(_axes_flat, _segments_ordered):
    for _scenario in _scenarios:
        _series = _ts_data[_scenario].get(seg)
        if _series is None:
            continue
        _valid = _series.dropna()
        ax.plot(
            _valid.index,
            _valid.values,
            label=_display(_scenario),
            color=_scenario_color[_scenario],
            linestyle=_scenario_ls[_scenario],
            linewidth=1.8,
            marker="o",
            markersize=3,
        )
    ax.set_title(seg, fontsize=fontsize - 1, pad=6)
    ax.set_xlabel("Year", fontsize=fontsize - 2)
    ax.set_ylabel("M€", fontsize=fontsize - 2)
    ax.tick_params(labelsize=fontsize - 3)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.spines[["top", "right"]].set_visible(False)

for ax in _axes_flat[len(_segments_ordered):]:
    ax.set_visible(False)

_handles, _labels = _axes_flat[0].get_legend_handles_labels()
fig.legend(_handles, _labels, loc="lower center", ncol=len(_scenarios),
           fontsize=fontsize - 1, frameon=False, bbox_to_anchor=(0.5, -0.02))
fig.suptitle("Total renovation subsidies over time by housing segment (Million €)",
             fontsize=fontsize + 1, y=1.01)
fig.tight_layout()
plt.show()


In [ ]:
run_folder = resolve_run("policy_assessment", run_name)
output_folder = run_folder / "img"
output_folder.mkdir(parents=True, exist_ok=True)

print(f"Run folder: {run_folder}")
print(f"Output folder: {output_folder}")

output_bundle = load_output_bundle(run_folder)
if not output_bundle:
    raise FileNotFoundError(f"No output.csv files found in {run_folder}")

print(f"Loaded output tables: {len(output_bundle)}")


In [ ]:
output_bundle = keep_scenarios(output_bundle, selected_scenarios)

if selected_scenarios:
    print(f"Filtered scenarios: {list(output_bundle.keys())}")

if not output_bundle:
    raise ValueError("Scenario filter removed all data. Update `selected_scenarios`.")

if scenario_renames:
    output_bundle = {scenario_renames.get(name, name): table for name, table in output_bundle.items()}
    optimal_scenario = scenario_renames.get(optimal_scenario, optimal_scenario)

def display_scenario(name):
    return scenario_title_labels.get(name, name)


In [ ]:
summaries = {}
parsed_by_metric = {}

for metric_name in metric_comparison:
    final_df = build_indicator_summary(
        output_bundle=output_bundle,
        pattern=indicator_pattern,
        select_value=metric_name,
        reference=reference_pattern,
        reference_multiplier=reference_scale,
    )

    if final_df.empty:
        raise ValueError(f"No indicators matched the configured regex pattern for metric '{metric_name}'.")

    summary_path = output_folder / f"subsidies_{metric_name}.csv"
    final_df.to_csv(summary_path, index=True)
    print(f"Saved wide-format summary: {summary_path}")

    summaries[metric_name] = final_df
    parsed_by_metric[metric_name] = parse_subsidy_indicator_index(final_df)

summaries[primary_metric]


## Metric comparison for subsidy-allocation gaps


This section is a sensitivity check on the policy-gap metric. For each non-optimal scenario, the gap relative to `OptimalSubsidies` is recomputed using four averaging windows from `output.csv`: the first 5, 10, 20, and 50 available years. The bars therefore compare how the measured subsidy-allocation gap changes when the averaging horizon changes, while keeping the same underlying scenario pair.


In [ ]:
scenarios = list(summaries[primary_metric].columns)
metric_labels = list(metric_comparison.values())

for scenario in [name for name in scenarios if name != optimal_scenario]:
    scenario_title = display_scenario(scenario)
    optimal_title = display_scenario(optimal_scenario)

    metric_delta_frames = []
    for metric_name, metric_label in metric_comparison.items():
        delta_df = subsidy_delta_relative_to_reference(
            parsed_by_metric[metric_name],
            reference_scenario=optimal_scenario,
        )
        scenario_delta = delta_df[delta_df["Scenario"] == scenario].copy()
        scenario_delta["Scenario"] = metric_label
        metric_delta_frames.append(scenario_delta)

    metric_delta_df = pd.concat(metric_delta_frames, ignore_index=True)

    save_eur = output_folder / f"subsidies_delta_metric_comparison_{scenario}_vs_{optimal_scenario}.png"
    plot_subsidy_delta_by_housing_status(
        metric_delta_df,
        scenario=metric_labels,
        optimal_scenario=optimal_title,
        label_size=fontsize,
        same_axis_limits=same_ylim,
        save=save_eur,
        suptitle=f"Subsidy allocation gap by owner income group and housing segment: {scenario_title} relative to {optimal_title} (metric comparison)",
        xlabel="",
        legend_position="bottom",
    )
    print(f"Saved: {save_eur}")

    save_pct = output_folder / f"subsidies_delta_pct_metric_comparison_{scenario}_vs_{optimal_scenario}.png"
    plot_subsidy_delta_pct_by_housing_status(
        metric_delta_df,
        scenario=metric_labels,
        optimal_scenario=optimal_title,
        label_size=fontsize,
        same_axis_limits=same_ylim,
        save=save_pct,
        suptitle=f"Relative subsidy allocation gap by owner income group and housing segment: {scenario_title} relative to {optimal_title} (metric comparison)",
        xlabel="",
        legend_position="bottom",
    )
    print(f"Saved: {save_pct}")


## Allocation and ad-valorem analysis based on the 5-year average


The remaining figures use the 5-year average only. This keeps the main policy comparison close to a short-run implementation view, while the previous section documents how sensitive the allocation-gap result is to the averaging horizon.


In [ ]:
primary_label = metric_comparison[primary_metric]
final_df = summaries[primary_metric]
parsed = parsed_by_metric[primary_metric]
scenarios = list(final_df.columns)
comparison_title = " vs. ".join(display_scenario(scenario) for scenario in scenarios)
title_suffix = f" ({primary_label})"

save_path = output_folder / f"subsidies_{primary_metric}_by_income_and_segment.png"
plot_subsidies_by_housing_status(
    parsed,
    scenarios=scenarios,
    label_size=fontsize,
    same_axis_limits=same_ylim,
    save=save_path,
    suptitle=f"Renovation subsidy allocation by owner income group and housing segment: {comparison_title}{title_suffix}",
    xlabel="",
)
print(f"Saved: {save_path}")

parsed_wide = parsed[parsed["Income"] != "Total"].pivot_table(
    index=["Housing", "Status", "Income"], columns="Scenario", values="Value",
)
parsed_wide_path = output_folder / f"subsidies_{primary_metric}_by_segment_income.csv"
parsed_wide.to_csv(parsed_wide_path)
print(f"Saved: {parsed_wide_path}")

ad_valorem_df = build_ad_valorem_ratio(
    output_bundle,
    subsidy_pattern=indicator_pattern,
    investment_pattern=investment_pattern,
    select_value=primary_metric,
)
av_path = output_folder / f"ad_valorem_ratio_{primary_metric}.csv"
(ad_valorem_df * 100).to_csv(av_path)
print(f"Saved: {av_path}")

save_path = output_folder / f"ad_valorem_ratio_{primary_metric}.png"
plot_ad_valorem_ratio_by_housing_status(
    ad_valorem_df,
    label_size=fontsize,
    same_axis_limits=same_ylim,
    save=save_path,
    suptitle=f"Implicit subsidy rate by housing segment: {comparison_title}{title_suffix}",
)
print(f"Saved: {save_path}")

hs_df = subsidy_summary_by_housing_status(parsed)
hs_path = output_folder / f"subsidies_{primary_metric}_by_housing_status.csv"
hs_df.to_csv(hs_path)
print(f"Saved: {hs_path}")

plot_indicator_bar_grid(
    hs_df,
    same_axis_limits=same_ylim,
    label_size=fontsize,
    xlabel="",
    value_formatter=lambda value: f"{value:,.0f} M€",
    title=f"Total renovation subsidies by housing segment: {comparison_title}{title_suffix}",
)

inc_df = subsidy_summary_by_income(parsed)
inc_path = output_folder / f"subsidies_{primary_metric}_by_income.csv"
inc_df.to_csv(inc_path)
print(f"Saved: {inc_path}")

plot_indicator_bar_grid(
    inc_df,
    same_axis_limits=same_ylim,
    label_size=fontsize,
    xlabel="",
    value_formatter=lambda value: f"{value:,.0f} M€",
    title=f"Total renovation subsidies by owner income group: {comparison_title}{title_suffix}",
)
